In [1]:
"""
ML_autoencoder_cnn.ipynb
---
Bus trip delay detection via Latent Feature Representation Learning.
Inspired by: Duncan & Chen, Computers & Security 128 (2023) 103138.
Adapted from the censorship-detection pipeline (ML_autoencoder_v4) to the
NetMob 2026 Niterói bus dataset.

Task: classify bus trips as DELAYED (median stop delay >= 5 min vs the GTFS
schedule) vs ON-TIME, using only label-free GPS-derived sequence features
(speed, progress, heading change, dwell, weather, headway, ...). Labels come
from 00_preprocess.ipynb (GPS-inferred stop arrivals vs GTFS stop_times).

Architecture
──────────
Phase 1 - Conv1D Autoencoder (UNSUPERVISED, fitted per fold on train split):
    encoder: stacked Conv1d blocks over (T=64, F=12) + 1x1 conv -> z_seq,
             length-masked mean -> pooled z (LATENT_DIM=64)
    decoder: z broadcast over T + learned positional embedding -> Conv1d -> recon
Phase 2 - frozen encoder; downstream supervised classifiers on the latents:
    AE+MLP / AE+XGBoost / AE+RF / AE+LR   on pooled z    (N, 64)
    AE+LSTM / AE+GRU                      on z_seq       (N, T, 64)

Cross-validation: StratifiedGroupKFold grouped by SERVICE DATE — day-level
effects (weather, traffic, telemetry quality) are the dominant leakage
channel, so every fold tests on entirely unseen days.

Extra export vs the reference pipeline: per-trip OUT-OF-FOLD reconstruction
error (each trip scored by the AE of the fold where it was held out) ->
anomaly_scores.csv, consumed by 03_anomaly_analysis.ipynb.
"""

'\nML_autoencoder_cnn.ipynb\n---\nBus trip delay detection via Latent Feature Representation Learning.\nInspired by: Duncan & Chen, Computers & Security 128 (2023) 103138.\nAdapted from the censorship-detection pipeline (ML_autoencoder_v4) to the\nNetMob 2026 Niterói bus dataset.\n\nTask: classify bus trips as DELAYED (median stop delay >= 5 min vs the GTFS\nschedule) vs ON-TIME, using only label-free GPS-derived sequence features\n(speed, progress, heading change, dwell, weather, headway, ...). Labels come\nfrom 00_preprocess.ipynb (GPS-inferred stop arrivals vs GTFS stop_times).\n\nArchitecture\n──────────\nPhase 1 - Conv1D Autoencoder (UNSUPERVISED, fitted per fold on train split):\n    encoder: stacked Conv1d blocks over (T=64, F=12) + 1x1 conv -> z_seq,\n             length-masked mean -> pooled z (LATENT_DIM=64)\n    decoder: z broadcast over T + learned positional embedding -> Conv1d -> recon\nPhase 2 - frozen encoder; downstream supervised classifiers on the latents:\n    AE+ML

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, log_loss,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE

import xgboost as xgb
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, TensorDataset
import warnings
import os

In [3]:
warnings.filterwarnings("ignore")

FEATURE_COLS = [
    "speed_mps", "accel", "heading_change", "progress_frac",
    "progress_rate", "cross_track_m", "dwell_frac",
    "tod_sin", "tod_cos", "is_weekend", "rain_mm", "headway_min",
]

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.set_num_threads(os.cpu_count())

dataset_in   = "./cache/sequences.npz"
meta_in      = "./cache/trips_meta.parquet"
artifact_dir = "./artifacts_ae_cnn"
os.makedirs(artifact_dir, exist_ok=True)

# ── sequence config (must match netmob_prep.py) ──
T_MAX = 64

# ── smoke-mode toggle: quick end-to-end sanity run ──
SMOKE = os.environ.get("NETMOB_SMOKE", "0") == "1"

# ── autoencoder hyper-params ──
LATENT_DIM = 64
AE_HIDDEN  = 128
AE_EPOCHS  = 3 if SMOKE else 20
AE_LR      = 1e-3
AE_BATCH   = 64

# ── Conv1D-AE hyper-params ──
CNN_KERNEL_SIZE = 5    # "same" padding = CNN_KERNEL_SIZE // 2
CNN_NUM_LAYERS  = 3    # stacked Conv1d blocks in encoder/decoder

# ── classifier hyper-params ──
CLS_EPOCHS = 3 if SMOKE else 12
CLS_BATCH  = 128
FOLDS      = 2 if SMOKE else 5
SMOKE_MAX_TRIPS = 3000
TSNE_MAX_POINTS = 4000   # subsample per fold — t-SNE is O(N^2) on CPU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}  |  SMOKE: {SMOKE}")
print(f"Latent dim: {LATENT_DIM}  |  AE epochs: {AE_EPOCHS}  |  CV folds: {FOLDS}")

Device: cpu  |  SMOKE: False
Latent dim: 64  |  AE epochs: 20  |  CV folds: 5


## ETL — load trip sequences

In [4]:
def load_trip_sequences(npz_path=dataset_in, meta_path=meta_in):
    """Load the cache produced by 00_preprocess.ipynb.
    Returns X_seq (N, T, F) float32, y, trip_ids, seq_lens, day_groups."""
    z = np.load(npz_path, allow_pickle=True)
    X_seq    = z["X"].astype(np.float32)
    y        = z["y"].astype(int)
    seq_lens = z["lengths"].astype(int)
    trip_ids = z["trip_instance_id"]
    groups   = z["groups"]                      # service_date per trip
    cached_cols = [str(c) for c in z["feature_cols"]]
    assert cached_cols == FEATURE_COLS, "feature layout drifted vs cache"
    meta = pd.read_parquet(meta_path)
    return X_seq, y, trip_ids, seq_lens, groups, meta


def print_stats(y, groups, meta):
    print("\n── Label summary ─────────────────────────────────────")
    print(f"  Trips        : {len(y):>7,}")
    print(f"  Delayed      : {int(y.sum()):>7,}  ({100 * y.mean():.1f}%)")
    print(f"  On-time      : {int((y == 0).sum()):>7,}")
    print(f"  Service days : {len(np.unique(groups))}")
    bal = (pd.DataFrame({"day": groups, "y": y})
           .groupby("day")["y"].agg(["mean", "size"]))
    print("\n  per-day delayed share:")
    for day, r in bal.iterrows():
        print(f"    {day}: {r['mean']:.2f}  (n={int(r['size'])})")

In [5]:
def scale_sequences(X_train_seq, X_test_seq):
    """Per-feature StandardScaler fitted on training data only (no leakage)."""
    N_tr, T, F  = X_train_seq.shape
    N_te        = X_test_seq.shape[0]
    scaler      = StandardScaler()
    X_tr_2d     = X_train_seq.reshape(N_tr * T, F)
    X_te_2d     = X_test_seq.reshape(N_te * T, F)
    X_tr_scaled = scaler.fit_transform(X_tr_2d).reshape(N_tr, T, F).astype(np.float32)
    X_te_scaled = scaler.transform(X_te_2d).reshape(N_te, T, F).astype(np.float32)
    return X_tr_scaled, X_te_scaled, scaler

## Conv1D Autoencoder

The encoder applies a stack of `CNN_NUM_LAYERS` Conv1d blocks (kernel size
`CNN_KERNEL_SIZE`, "same" padding) over the time axis of `(T=64, F=12)`,
followed by a 1x1 conv projecting to `LATENT_DIM=64` channels. This directly
gives the per-timestep latent sequence `z_seq` of shape `(T=64, 64)`.
The pooled latent `z` (N, LATENT_DIM) is the length-masked mean of `z_seq`
over the real (non-padded) timesteps.

The decoder cannot rely on recurrence to vary its output over time, so it
broadcasts `z` across all T positions, adds a learned positional embedding,
and runs Conv1d blocks to reconstruct the input sequence.

In [6]:
class ConvEncoder(nn.Module):
    """
    Conv1d encoder: (N, T, F) -> per-timestep latent z_seq (N, T, LATENT_DIM)
                              -> pooled latent z (N, LATENT_DIM)
    Stacked Conv1d blocks with "same" padding preserve T throughout;
    a final 1x1 conv projects channels -> LATENT_DIM, giving z_seq
    directly. z is the length-masked mean of z_seq over the real
    (non-padded) timesteps, so padding never influences z.
    """
    def __init__(self, input_size, hidden_size=AE_HIDDEN,
                 latent_dim=LATENT_DIM, num_layers=CNN_NUM_LAYERS,
                 kernel_size=CNN_KERNEL_SIZE, dropout=0.2):
        super().__init__()
        pad = kernel_size // 2
        blocks, in_ch = [], input_size
        for _ in range(num_layers):
            blocks += [
                nn.Conv1d(in_ch, hidden_size, kernel_size, padding=pad),
                nn.BatchNorm1d(hidden_size),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            in_ch = hidden_size
        self.conv = nn.Sequential(*blocks)
        self.proj = nn.Conv1d(hidden_size, latent_dim, kernel_size=1)

    def forward(self, x, lengths=None):
        h     = self.conv(x.transpose(1, 2))   # (N, hidden, T)
        z_seq = self.proj(h).transpose(1, 2)   # (N, T, latent_dim)
        if lengths is not None:
            t_idx = torch.arange(z_seq.size(1), device=z_seq.device).unsqueeze(0)
            mask  = (t_idx < lengths.unsqueeze(1).to(z_seq.device)).float().unsqueeze(2)
            z     = (z_seq * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        else:
            z = z_seq.mean(dim=1)
        return z, z_seq


class ConvDecoder(nn.Module):
    """
    Conv1d decoder: z (N, LATENT_DIM) -> reconstructed (N, T, F)
    Conv1d alone is translation-invariant, so a constant z broadcast over T
    would decode to (almost) the same output at every timestep. A learned
    positional embedding added to the broadcast z is what lets the decoder
    reconstruct a time-varying sequence from a single bottleneck vector.
    Fixed-length output — MSE loss requires symmetric shapes.
    """
    def __init__(self, latent_dim=LATENT_DIM, hidden_size=AE_HIDDEN,
                 output_size=None, seq_len=T_MAX,
                 num_layers=CNN_NUM_LAYERS, kernel_size=CNN_KERNEL_SIZE,
                 dropout=0.2):
        super().__init__()
        self.seq_len = seq_len
        self.pos_emb = nn.Parameter(torch.zeros(1, latent_dim, seq_len))
        nn.init.normal_(self.pos_emb, std=0.02)
        pad = kernel_size // 2
        blocks, in_ch = [], latent_dim
        for _ in range(num_layers):
            blocks += [
                nn.Conv1d(in_ch, hidden_size, kernel_size, padding=pad),
                nn.BatchNorm1d(hidden_size),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            in_ch = hidden_size
        self.conv = nn.Sequential(*blocks)
        self.out = nn.Conv1d(hidden_size, output_size, kernel_size=1)

    def forward(self, z):
        z_rep = z.unsqueeze(2).expand(-1, -1, self.seq_len) + self.pos_emb
        h     = self.conv(z_rep)
        return self.out(h).transpose(1, 2)  # (N, T, F)


class ConvAutoencoder(nn.Module):
    def __init__(self, input_size, hidden_size=AE_HIDDEN,
                 latent_dim=LATENT_DIM, seq_len=T_MAX,
                 num_layers=CNN_NUM_LAYERS, kernel_size=CNN_KERNEL_SIZE,
                 dropout=0.2):
        super().__init__()
        self.encoder = ConvEncoder(input_size, hidden_size, latent_dim,
                                    num_layers, kernel_size, dropout)
        self.decoder = ConvDecoder(latent_dim, hidden_size, input_size, seq_len,
                                    num_layers, kernel_size, dropout)

    def forward(self, x, lengths=None):
        z, z_seq = self.encoder(x, lengths)
        recon    = self.decoder(z)
        return recon, z, z_seq

In [7]:
def train_autoencoder(X_train_seq, X_val_seq, lens_train=None, lens_val=None,
                      epochs=AE_EPOCHS, batch_size=AE_BATCH, lr=AE_LR):
    """Fit ConvAutoencoder on training sequences only (unsupervised)."""
    n_feat    = X_train_seq.shape[2]
    ae        = ConvAutoencoder(input_size=n_feat).to(device)
    opt       = torch.optim.Adam(ae.parameters(), lr=lr, weight_decay=1e-5)
    sched     = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    opt, mode="min", patience=6, factor=0.5)
    criterion = nn.MSELoss()

    X_tr_t = torch.tensor(X_train_seq).to(device)
    X_va_t = torch.tensor(X_val_seq).to(device)
    l_tr_t = torch.tensor(lens_train, dtype=torch.long).to(device) if lens_train is not None else None
    l_va_t = torch.tensor(lens_val,   dtype=torch.long).to(device) if lens_val   is not None else None

    dummy_lens = torch.zeros(len(X_tr_t), dtype=torch.long)
    ds     = TensorDataset(X_tr_t, l_tr_t if l_tr_t is not None else dummy_lens)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    use_lengths = l_tr_t is not None

    best_val, best_wts  = float("inf"), None
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        ae.train()
        ep_loss, nb = 0.0, 0
        for xb, lb in loader:
            opt.zero_grad()
            recon, _, _ = ae(xb, lb if use_lengths else None)
            loss     = criterion(recon, xb)
            loss.backward()
            nn.utils.clip_grad_norm_(ae.parameters(), 1.0)
            opt.step()
            ep_loss += loss.item(); nb += 1
        train_losses.append(ep_loss / max(1, nb))

        ae.eval()
        with torch.no_grad():
            recon_val, _, _ = ae(X_va_t, l_va_t if use_lengths else None)
            vl = criterion(recon_val, X_va_t).item()
        val_losses.append(vl)
        sched.step(vl)
        if vl < best_val:
            best_val = vl
            best_wts = {k: v.clone() for k, v in ae.state_dict().items()}
        if (epoch + 1) % 10 == 0:
            print(f"      AE epoch {epoch+1:3d}/{epochs}"
                  f" | train MSE: {train_losses[-1]:.5f}"
                  f" | val MSE: {vl:.5f}")

    ae.load_state_dict(best_wts)
    return ae, train_losses, val_losses


def extract_latent(ae, X_seq, lens=None, batch_size=256):
    """Run frozen encoder over X_seq. Returns (z, z_seq)."""
    ae.eval()
    z_parts, zseq_parts = [], []
    with torch.no_grad():
        for i in range(0, len(X_seq), batch_size):
            xb = torch.tensor(X_seq[i:i+batch_size]).to(device)
            lb = torch.tensor(lens[i:i+batch_size], dtype=torch.long).to(device) \
                 if lens is not None else None
            z, z_seq = ae.encoder(xb, lb)
            z_parts.append(z.cpu().numpy())
            zseq_parts.append(z_seq.cpu().numpy())
    return np.vstack(z_parts), np.vstack(zseq_parts)


def reconstruction_errors(ae, X_seq, lens, batch_size=256):
    """Per-trip reconstruction MSE masked to the true (non-padded) length —
    the anomaly score used by 03_anomaly_analysis.ipynb."""
    ae.eval()
    errs = []
    with torch.no_grad():
        for i in range(0, len(X_seq), batch_size):
            xb = torch.tensor(X_seq[i:i+batch_size]).to(device)
            lb = torch.tensor(lens[i:i+batch_size], dtype=torch.long).to(device)
            recon, _, _ = ae(xb, lb)
            se   = (recon - xb) ** 2                       # (n, T, F)
            t_idx = torch.arange(se.size(1), device=device).unsqueeze(0)
            mask  = (t_idx < lb.unsqueeze(1)).float().unsqueeze(2)
            per_trip = (se * mask).sum(dim=(1, 2)) / (mask.sum(dim=(1, 2)) * se.size(2)).clamp(min=1.0)
            errs.append(per_trip.cpu().numpy())
    return np.concatenate(errs)

In [8]:
class MLPClassifier(nn.Module):
    """3-layer MLP operating on latent z. Input=LATENT_DIM, output=1 logit."""
    def __init__(self, input_dim=LATENT_DIM, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(1)


def train_mlp_on_z(z_train, y_train, z_val, y_val,
                   epochs=CLS_EPOCHS, batch_size=CLS_BATCH, pos_weight_val=1.0):
    """Train MLP on latent z. Returns (val_probs, train_losses, val_losses)."""
    model = MLPClassifier().to(device)
    pw    = torch.tensor([pos_weight_val], dtype=torch.float32).to(device)
    crit  = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                opt, mode="min", patience=5, factor=0.5)
    z_tr_t = torch.tensor(z_train, dtype=torch.float32).to(device)
    y_tr_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    z_va_t = torch.tensor(z_val,   dtype=torch.float32).to(device)
    y_va_t = torch.tensor(y_val,   dtype=torch.float32).to(device)
    ds     = TensorDataset(z_tr_t, y_tr_t)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    best_val, best_wts  = float("inf"), None
    tr_losses, va_losses = [], []
    for epoch in range(epochs):
        model.train()
        ep_loss, nb = 0.0, 0
        for zb, yb in loader:
            opt.zero_grad()
            loss = crit(model(zb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_loss += loss.item(); nb += 1
        tr_losses.append(ep_loss / max(1, nb))
        model.eval()
        with torch.no_grad():
            vl = crit(model(z_va_t), y_va_t).item()
        va_losses.append(vl); sched.step(vl)
        if vl < best_val:
            best_val = vl
            best_wts = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_wts); model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(z_va_t)).cpu().numpy()
    return probs, tr_losses, va_losses

In [9]:
# ── AE+LSTM / AE+GRU classifiers on per-timestep latent sequence z_seq ─────
# Both accept lengths and use pack_padded_sequence; input_size is LATENT_DIM.

class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Sequential(nn.Dropout(dropout),
                                   nn.Linear(hidden_size, 32), nn.ReLU(),
                                   nn.Linear(32, 1))

    def forward(self, x, lengths=None):
        if lengths is not None:
            lengths_cpu = lengths.clamp(min=1).cpu()
            packed = pack_padded_sequence(x, lengths_cpu, batch_first=True,
                                          enforce_sorted=False)
            out_packed, _ = self.lstm(packed)
            out, _ = pad_packed_sequence(out_packed, batch_first=True)
            idx  = (lengths_cpu - 1).clamp(min=0).to(out.device)
            last = out[torch.arange(out.size(0), device=out.device), idx]
        else:
            out, _ = self.lstm(x)
            last   = out[:, -1, :]
        return self.head(last).squeeze(1)


class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=num_layers,
                          batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Sequential(nn.Dropout(dropout),
                                   nn.Linear(hidden_size, 32), nn.ReLU(),
                                   nn.Linear(32, 1))

    def forward(self, x, lengths=None):
        if lengths is not None:
            lengths_cpu = lengths.clamp(min=1).cpu()
            packed = pack_padded_sequence(x, lengths_cpu, batch_first=True,
                                          enforce_sorted=False)
            out_packed, _ = self.gru(packed)
            out, _ = pad_packed_sequence(out_packed, batch_first=True)
            idx  = (lengths_cpu - 1).clamp(min=0).to(out.device)
            last = out[torch.arange(out.size(0), device=out.device), idx]
        else:
            out, _ = self.gru(x)
            last   = out[:, -1, :]
        return self.head(last).squeeze(1)


def train_rnn_classifier(model, X_train, y_train, X_val, y_val,
                       lens_train=None, lens_val=None,
                       epochs=CLS_EPOCHS, batch_size=CLS_BATCH, pos_weight_val=1.0):
    """Train LSTM or GRU classifier on a latent sequence z_seq."""
    model  = model.to(device)
    pw     = torch.tensor([pos_weight_val], dtype=torch.float32).to(device)
    crit   = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt    = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched  = torch.optim.lr_scheduler.ReduceLROnPlateau(
                 opt, mode="min", patience=5, factor=0.5)
    X_tr_t = torch.tensor(X_train).to(device)
    y_tr_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_va_t = torch.tensor(X_val).to(device)
    y_va_t = torch.tensor(y_val,  dtype=torch.float32).to(device)
    l_tr_t = torch.tensor(lens_train, dtype=torch.long).to(device) if lens_train is not None else None
    l_va_t = torch.tensor(lens_val,   dtype=torch.long).to(device) if lens_val   is not None else None

    dummy  = torch.zeros(len(X_tr_t), dtype=torch.long)
    ds     = TensorDataset(X_tr_t, y_tr_t, l_tr_t if l_tr_t is not None else dummy)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    use_l  = l_tr_t is not None

    best_val, best_wts  = float("inf"), None
    tr_losses, va_losses = [], []
    for epoch in range(epochs):
        model.train()
        ep_loss, nb = 0.0, 0
        for xb, yb, lb in loader:
            opt.zero_grad()
            loss = crit(model(xb, lb if use_l else None), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); ep_loss += loss.item(); nb += 1
        tr_losses.append(ep_loss / max(1, nb))
        model.eval()
        with torch.no_grad():
            vl = crit(model(X_va_t, l_va_t if use_l else None), y_va_t).item()
        va_losses.append(vl); sched.step(vl)
        if vl < best_val:
            best_val = vl
            best_wts = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_wts); model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(X_va_t, l_va_t if use_l else None)).cpu().numpy()
    return probs, tr_losses, va_losses

In [10]:
METRIC_NAMES = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
MODEL_NAMES  = ["AE+MLP", "AE+XGBoost", "AE+RF", "AE+LR", "AE+LSTM", "AE+GRU"]


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "accuracy" : accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall"   : recall_score(y_true, y_pred, zero_division=0),
        "f1"       : f1_score(y_true, y_pred, zero_division=0),
        "roc_auc"  : roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan,
        "pr_auc"   : average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan,
    }

## Cross-validation loop

For each fold:
1. Scale sequences (fit scaler on train split only).
2. **Train Conv1D Autoencoder** on `X_tr_seq` — no labels, pure MSE reconstruction.
3. **Freeze encoder**, discard decoder, extract pooled latent `z` and
   per-timestep latent sequence `z_seq` for train and test.
4. Train AE+MLP, AE+XGBoost, AE+RF, AE+LR on `z`.
5. Train AE+LSTM, AE+GRU on `z_seq`.
6. Evaluate all six models on the held-out fold, log metrics + loss curves,
   and record **out-of-fold reconstruction errors** as anomaly scores.

In [11]:
def run_cv(X_seq, y, trip_ids, seq_lens, day_groups, n_folds=FOLDS):
    """
    Full cross-validation pipeline, StratifiedGroupKFold by SERVICE DATE so
    every fold tests on unseen days (weather/traffic/telemetry leakage-safe).

    Each completed fold is checkpointed to {artifact_dir}/cv_fold<k>.pkl and
    restored on re-run, so an interrupted execution resumes at the first
    unfinished fold instead of retraining from scratch. Fold milestones are
    appended to {artifact_dir}/progress.log for external monitoring.

    Returns: results, val_curves, ae_curves, latent_store, anomaly_rows
    """
    import pickle
    import time

    def log(msg):
        line = f"{time.strftime('%H:%M:%S')} {msg}"
        print(line)
        with open(f"{artifact_dir}/progress.log", "a") as fh:
            fh.write(line + "\n")

    day_arr = np.asarray(day_groups)
    splitter = StratifiedGroupKFold(n_splits=n_folds, shuffle=True,
                                    random_state=RANDOM_SEED)
    log(f"CV start — {len(np.unique(day_arr))} service days, {n_folds} folds, "
        f"N={len(X_seq)}")

    results      = {m: [] for m in MODEL_NAMES}
    val_curves   = {m: {"train": [], "val": []} for m in MODEL_NAMES}
    ae_curves    = {"train": [], "val": []}
    latent_store = {}
    anomaly_rows = []

    for fold_idx, (tr_idx, te_idx) in enumerate(
        splitter.split(X_seq.reshape(len(X_seq), -1), y, groups=day_arr)
    ):
        ckpt_path = f"{artifact_dir}/cv_fold{fold_idx + 1}.pkl"
        if os.path.exists(ckpt_path):
            with open(ckpt_path, "rb") as fh:
                fold = pickle.load(fh)
            log(f"FOLD {fold_idx+1}/{n_folds} restored from checkpoint")
        else:
            log(f"FOLD {fold_idx+1}/{n_folds} start | "
                f"train delayed {int(y[tr_idx].sum())} | "
                f"test delayed {int(y[te_idx].sum())} | "
                f"test days {sorted(set(day_arr[te_idx]))}")
            fold = _run_fold(X_seq, y, trip_ids, seq_lens, tr_idx, te_idx, log)
            with open(ckpt_path, "wb") as fh:
                pickle.dump(fold, fh)
            log(f"FOLD {fold_idx+1}/{n_folds} done -> {ckpt_path}")

        ae_curves["train"].append(fold["ae_tr"])
        ae_curves["val"].append(fold["ae_va"])
        latent_store[fold_idx] = fold["latents"]
        anomaly_rows.extend(fold["anomaly_rows"])
        for m in MODEL_NAMES:
            results[m].append(fold["metrics"][m])
            val_curves[m]["train"].append(fold["curves"][m]["train"])
            val_curves[m]["val"].append(fold["curves"][m]["val"])

    log("CV complete")
    return results, val_curves, ae_curves, latent_store, anomaly_rows


def _run_fold(X_seq, y, trip_ids, seq_lens, tr_idx, te_idx, log):
    """Train AE + all six downstream classifiers for one fold; returns a
    picklable dict with everything the reporting stage needs."""
    # ── 1. Scale ──────────────────────────────────────────────────────
    X_tr_seq, X_te_seq, _ = scale_sequences(X_seq[tr_idx], X_seq[te_idx])
    y_tr, y_te = y[tr_idx], y[te_idx]
    lens_tr    = seq_lens[tr_idx]
    lens_te    = seq_lens[te_idx]
    n_neg      = (y_tr == 0).sum()
    n_pos      = max(1, (y_tr == 1).sum())
    pos_weight = n_neg / n_pos
    cw         = {0: 1.0, 1: pos_weight}
    metrics, curves = {}, {}

    # ── 2. Autoencoder pre-training (no labels) ───────────────────────
    log("  [Phase 1] training autoencoder...")
    ae, ae_tr, ae_va = train_autoencoder(
        X_tr_seq, X_te_seq, lens_train=lens_tr, lens_val=lens_te)
    log(f"    final AE MSE train={ae_tr[-1]:.5f} val={ae_va[-1]:.5f}")

    # ── 2b. Out-of-fold anomaly scores (test split only) ──────────────
    oof_err = reconstruction_errors(ae, X_te_seq, lens_te)
    anomaly_rows = [
        {"trip_instance_id": tid, "recon_mse": float(e), "y": int(yy)}
        for tid, e, yy in zip(trip_ids[te_idx], oof_err, y_te)]

    # ── 3. Extract latent z and z_seq ─────────────────────────────────
    z_train, zseq_train = extract_latent(ae, X_tr_seq, lens=lens_tr)
    z_test,  zseq_test  = extract_latent(ae, X_te_seq, lens=lens_te)
    latents = {"z_test": z_test, "y_test": y_te,
               "z_train": z_train, "y_train": y_tr,
               "trip_ids_test": trip_ids[te_idx]}

    # ── 4a. AE + MLP ──────────────────────────────────────────────────
    log("  training AE+MLP...")
    probs, tr_c, va_c = train_mlp_on_z(
        z_train, y_tr, z_test, y_te, pos_weight_val=pos_weight)
    metrics["AE+MLP"] = compute_metrics(y_te, probs)
    curves["AE+MLP"] = {"train": tr_c, "val": va_c}

    # ── 4b. AE + XGBoost ──────────────────────────────────────────────
    log("  training AE+XGBoost...")
    xgb_model = xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=pos_weight,
        eval_metric="logloss",
        random_state=RANDOM_SEED, verbosity=0)
    xgb_model.fit(z_train, y_tr,
                  eval_set=[(z_train, y_tr), (z_test, y_te)], verbose=False)
    ev = xgb_model.evals_result()
    metrics["AE+XGBoost"] = compute_metrics(
        y_te, xgb_model.predict_proba(z_test)[:, 1])
    curves["AE+XGBoost"] = {"train": ev["validation_0"]["logloss"],
                            "val": ev["validation_1"]["logloss"]}

    # ── 4c. AE + Random Forest ────────────────────────────────────────
    log("  training AE+RF...")
    rf_model = RandomForestClassifier(
        n_estimators=300, max_depth=12, class_weight=cw,
        random_state=RANDOM_SEED, n_jobs=-1)
    rf_model.fit(z_train, y_tr)
    rf_probs = rf_model.predict_proba(z_test)[:, 1]
    metrics["AE+RF"] = compute_metrics(y_te, rf_probs)
    curves["AE+RF"] = {
        "train": [log_loss(y_tr, rf_model.predict_proba(z_train)[:, 1])],
        "val":   [log_loss(y_te, rf_probs)]}

    # ── 4d. AE + Logistic Regression ──────────────────────────────────
    log("  training AE+LR...")
    lr_model = LogisticRegression(
        C=1.0, class_weight=cw, solver="lbfgs",
        max_iter=1000, random_state=RANDOM_SEED)
    lr_model.fit(z_train, y_tr)
    lr_probs = lr_model.predict_proba(z_test)[:, 1]
    metrics["AE+LR"] = compute_metrics(y_te, lr_probs)
    curves["AE+LR"] = {
        "train": [log_loss(y_tr, lr_model.predict_proba(z_train)[:, 1])],
        "val":   [log_loss(y_te, lr_probs)]}

    # ── 5. AE+LSTM / AE+GRU on latent sequence z_seq ──────────────────
    latent_dim = zseq_train.shape[2]
    for name, cls in [("AE+LSTM", LSTMClassifier), ("AE+GRU", GRUClassifier)]:
        log(f"  training {name} (latent sequence)...")
        probs, tr_c, va_c = train_rnn_classifier(
            cls(input_size=latent_dim), zseq_train, y_tr, zseq_test, y_te,
            lens_train=lens_tr, lens_val=lens_te, pos_weight_val=pos_weight)
        metrics[name] = compute_metrics(y_te, probs)
        curves[name] = {"train": tr_c, "val": va_c}

    for name in MODEL_NAMES:
        log(f"    {name:<11} F1 {metrics[name]['f1']:.3f} "
            f"ROC-AUC {metrics[name]['roc_auc']:.3f}")
    return {"ae_tr": ae_tr, "ae_va": ae_va, "latents": latents,
            "anomaly_rows": anomaly_rows, "metrics": metrics,
            "curves": curves}

## Reporting & Visualisation

In [12]:
def summarize_results(results):
    rows = []
    for model_name, fold_results in results.items():
        for fold_i, metrics in enumerate(fold_results):
            row = {"model": model_name, "fold": fold_i + 1}
            row.update(metrics); rows.append(row)
        means = {m: np.mean([f[m] for f in fold_results]) for m in METRIC_NAMES}
        stds  = {f"{m}_std": np.std([f[m] for f in fold_results]) for m in METRIC_NAMES}
        row   = {"model": model_name, "fold": "mean"}
        row.update(means); row.update(stds); rows.append(row)
    return pd.DataFrame(rows)


def print_summary_table(summary_df):
    print("\n" + "=" * 70)
    print("  RESULTS SUMMARY (mean +/- std across folds)")
    print("=" * 70)
    means = summary_df[summary_df["fold"] == "mean"]
    for _, row in means.iterrows():
        print(f"\n  {row['model']}")
        for m in METRIC_NAMES:
            print(f"    {m:<12}: {row[m]:.4f} +/- {row.get(m+'_std', 0):.4f}")
    print("=" * 70)

In [13]:
def plot_results(summary_df, output_path):
    """Bar chart: all models side by side across 5 metrics."""
    means   = summary_df[summary_df["fold"] == "mean"]
    models  = means["model"].tolist()
    metrics = ["precision", "recall", "f1", "roc_auc", "pr_auc"]
    labels  = ["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]
    x       = np.arange(len(metrics))
    n_mod   = len(models)
    width   = 0.80 / n_mod
    palette = ["#1D9E75", "#2196F3", "#FF9800", "#9C27B0", "#607D8B", "#78909C"]
    fig, ax = plt.subplots(figsize=(16, 7))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")
    for i, (model, color) in enumerate(zip(models, palette)):
        row    = means[means["model"] == model].iloc[0]
        vals   = [row[m] for m in metrics]
        errs   = [row.get(f"{m}_std", 0) for m in metrics]
        offset = (i - n_mod / 2 + 0.5) * width
        bars   = ax.bar(x + offset, vals, width, label=model, color=color,
                        alpha=0.88, yerr=errs, capsize=3,
                        error_kw={"color": "#555555", "alpha": 0.5})
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.012,
                    f"{val:.2f}", ha="center", va="bottom",
                    fontsize=6.5, color="black", fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(labels, color="black", fontsize=11)
    ax.set_ylim(0, 1.22)
    ax.set_ylabel("Score", color="black", fontsize=12)
    ax.set_title("Bus delay detection — AE latent-space classifiers\n"
                 "(Day-grouped Stratified 5-Fold CV, mean +/- std)",
                 color="black", fontsize=13, pad=15)
    ax.tick_params(colors="black")
    ax.spines[:].set_color("#bbbbbb")
    ax.yaxis.grid(True, color="#dddddd", linestyle="--", alpha=0.6)
    ax.set_axisbelow(True)
    ax.legend(facecolor="white", edgecolor="#bbbbbb", labelcolor="black",
              fontsize=9, ncol=3, loc="upper right")
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  Comparison plot saved -> {output_path}")

In [14]:
def plot_ae_reconstruction(ae_curves, output_path):
    """One subplot per fold: AE train/val reconstruction MSE over epochs."""
    n_folds = len(ae_curves["train"])
    fig, axes = plt.subplots(1, n_folds, figsize=(5 * n_folds, 4), sharey=True)
    fig.patch.set_facecolor("white")
    if n_folds == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        ax.set_facecolor("white")
        ax.tick_params(colors="black", labelsize=8)
        ax.spines[:].set_color("#bbbbbb")
        ax.yaxis.grid(True, color="#dddddd", linestyle="--", alpha=0.5)
        tr  = ae_curves["train"][i]
        va  = ae_curves["val"][i]
        eps = range(1, len(tr) + 1)
        ax.plot(eps, tr, color="#1D9E75", linewidth=2, label="Train MSE")
        ax.plot(eps, va, color="#FF6B6B", linewidth=2, label="Val MSE")
        ax.set_title(f"Fold {i+1}", color="black", fontsize=11)
        ax.set_xlabel("Epoch", color="black", fontsize=9)
        if i == 0:
            ax.set_ylabel("Reconstruction MSE", color="black", fontsize=9)
        ax.legend(facecolor="white", edgecolor="#bbbbbb",
                  labelcolor="black", fontsize=8)
    fig.suptitle("Conv1D Autoencoder — Reconstruction Loss per Fold",
                 color="black", fontsize=13)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  AE reconstruction curves saved -> {output_path}")

In [15]:
def plot_validation_curves(val_curves, output_path):
    """Train vs val loss curves for all downstream classifiers."""
    SINGLE = {"AE+RF", "AE+LR"}
    palette = {
        "AE+MLP": "#1D9E75", "AE+XGBoost": "#2196F3",
        "AE+RF":  "#FF9800", "AE+LR":      "#9C27B0",
        "AE+LSTM": "#607D8B", "AE+GRU":     "#78909C",
    }
    x_label = {
        "AE+MLP": "Epoch", "AE+XGBoost": "Boosting round",
        "AE+LSTM": "Epoch", "AE+GRU": "Epoch",
    }
    fig, axes = plt.subplots(2, 3, figsize=(18, 9))
    fig.patch.set_facecolor("white")
    for ax_idx, model in enumerate(MODEL_NAMES):
        ax    = axes.flatten()[ax_idx]
        color = palette[model]
        ax.set_facecolor("white")
        ax.tick_params(colors="black", labelsize=8)
        ax.spines[:].set_color("#bbbbbb")
        ax.yaxis.grid(True, color="#dddddd", linestyle="--", alpha=0.5)
        ax.set_axisbelow(True)
        tr_folds = val_curves[model]["train"]
        va_folds = val_curves[model]["val"]
        if model in SINGLE:
            tr_pts = [f[0] for f in tr_folds]
            va_pts = [f[0] for f in va_folds]
            ax.scatter([1]*len(tr_pts), tr_pts, color=color,     alpha=0.5, s=60)
            ax.scatter([1]*len(va_pts), va_pts, color="#FF6B6B",  alpha=0.5, s=60)
            ax.scatter([1], [np.mean(tr_pts)], color=color,      s=130, zorder=5,
                       label=f"Train mean={np.mean(tr_pts):.3f}")
            ax.scatter([1], [np.mean(va_pts)], color="#FF6B6B",   s=130, zorder=5,
                       label=f"Val mean={np.mean(va_pts):.3f}")
            ax.set_xticks([])
        else:
            for fold_tr, fold_va in zip(tr_folds, va_folds):
                xs = range(1, len(fold_tr) + 1)
                ax.plot(xs, fold_tr, color=color,     alpha=0.2, linewidth=1)
                ax.plot(xs, fold_va, color="#FF6B6B",  alpha=0.2, linewidth=1)
            min_len = min(len(f) for f in tr_folds)
            mean_tr = np.mean([f[:min_len] for f in tr_folds], axis=0)
            mean_va = np.mean([f[:min_len] for f in va_folds], axis=0)
            xs = range(1, len(mean_tr) + 1)
            ax.plot(xs, mean_tr, color=color,     linewidth=2.5, label="Train (mean)")
            ax.plot(xs, mean_va, color="#FF6B6B",  linewidth=2.5, label="Val (mean)")
            ax.set_xlabel(x_label.get(model, "Step"), color="black", fontsize=8)
        ax.set_ylabel("Loss", color="black", fontsize=8)
        ax.set_title(model, color=color, fontsize=11, fontweight="bold")
        ax.legend(facecolor="white", edgecolor="#bbbbbb",
                  labelcolor="black", fontsize=7, loc="upper right")
    fig.suptitle("Validation curves — train vs val loss (all classifiers)\n"
                 "Bold = mean across folds  |  Translucent = individual folds",
                 color="black", fontsize=13)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  Validation curves saved -> {output_path}")

In [16]:
def plot_latent_tsne(latent_store, output_path, max_points=TSNE_MAX_POINTS):
    """
    t-SNE grid: one subplot per fold, coloured by delay label.
    AE was trained without labels — colour added post-hoc. Points are
    subsampled per fold (t-SNE is O(N^2) and this runs on CPU).
    """
    import sklearn
    sk_version = tuple(int(x) for x in sklearn.__version__.split(".")[:2])

    n_folds = len(latent_store)
    fig, axes = plt.subplots(1, n_folds, figsize=(6 * n_folds, 5))
    fig.patch.set_facecolor("white")
    if n_folds == 1:
        axes = [axes]

    rng = np.random.default_rng(RANDOM_SEED)
    for fold_idx, ax in enumerate(axes):
        z_all = np.vstack([latent_store[fold_idx]["z_train"],
                           latent_store[fold_idx]["z_test"]])
        y_all = np.concatenate([latent_store[fold_idx]["y_train"],
                                latent_store[fold_idx]["y_test"]])
        if len(z_all) > max_points:
            sel = rng.choice(len(z_all), max_points, replace=False)
            z_all, y_all = z_all[sel], y_all[sel]

        print(f"  Fold {fold_idx+1}: t-SNE on {len(z_all)} vectors...")
        tsne_kwargs = dict(n_components=2, perplexity=40,
                           random_state=RANDOM_SEED,
                           learning_rate="auto", init="pca")
        tsne_kwargs["max_iter" if sk_version >= (1, 5) else "n_iter"] = 1000
        z_2d = TSNE(**tsne_kwargs).fit_transform(z_all)

        ax.set_facecolor("white")
        ax.tick_params(colors="black", labelsize=7)
        ax.spines[:].set_color("#bbbbbb")

        for label, color, name in [(0, "#378ADD", "On-time"),
                                   (1, "#D85A30", "Delayed")]:
            mask = y_all == label
            ax.scatter(z_2d[mask, 0], z_2d[mask, 1],
                       c=color, s=5, alpha=0.5 if label == 0 else 0.85,
                       label=f"{name} (n={mask.sum()})", linewidths=0)

        ax.set_title(f"Fold {fold_idx+1}", color="black",
                     fontsize=11, fontweight="bold")
        ax.set_xlabel("t-SNE 1", color="black", fontsize=8)
        if fold_idx == 0:
            ax.set_ylabel("t-SNE 2", color="black", fontsize=8)
        ax.legend(facecolor="white", edgecolor="#bbbbbb",
                  labelcolor="black", fontsize=7, markerscale=3,
                  loc="upper right")

    fig.suptitle(
        "t-SNE of latent z — all folds\n"
        "AE trained without labels; colours added post-hoc",
        color="black", fontsize=13, y=1.02,
    )
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  t-SNE grid saved -> {output_path}")

## Run

In [17]:
SEP = "=" * 64
print(f"\n{SEP}")
print("  Bus Delay Detection via Autoencoder Latent Feature Learning")
print("  Conv1D AE; ALL classifiers run on AE latent representations")
print(SEP)
print(f"  Input       : {dataset_in}")
print(f"  Latent dim  : {LATENT_DIM}")
print(f"  AE epochs   : {AE_EPOCHS}")
print(f"  CLS epochs  : {CLS_EPOCHS}")
print(f"  CV folds    : {FOLDS}")
print(f"  Device      : {device}")
print(f"{SEP}\n")


  Bus Delay Detection via Autoencoder Latent Feature Learning
  Conv1D AE; ALL classifiers run on AE latent representations
  Input       : ./cache/sequences.npz
  Latent dim  : 64
  AE epochs   : 20
  CLS epochs  : 12
  CV folds    : 5
  Device      : cpu



In [18]:
X_seq, y, trip_ids, seq_lens, groups, meta = load_trip_sequences()
if SMOKE and len(X_seq) > SMOKE_MAX_TRIPS:
    rng = np.random.default_rng(RANDOM_SEED)
    sel = np.sort(rng.choice(len(X_seq), SMOKE_MAX_TRIPS, replace=False))
    X_seq, y, trip_ids, seq_lens, groups = (
        X_seq[sel], y[sel], trip_ids[sel], seq_lens[sel], groups[sel])
    print(f"[SMOKE] subsampled to {len(X_seq)} trips")
print(f"X_seq: {X_seq.shape}  seq_lens: {seq_lens.min()}–{seq_lens.max()}")
print_stats(y, groups, meta)

X_seq: (22605, 64, 12)  seq_lens: 9–64

── Label summary ─────────────────────────────────────
  Trips        :  22,605
  Delayed      :   4,730  (20.9%)
  On-time      :  17,875
  Service days : 20

  per-day delayed share:
    2026-03-10: 0.30  (n=23)
    2026-03-11: 0.30  (n=1447)
    2026-03-12: 0.24  (n=1417)
    2026-03-13: 0.21  (n=1584)
    2026-03-14: 0.20  (n=1420)
    2026-03-15: 0.11  (n=1027)
    2026-03-16: 0.23  (n=781)
    2026-03-17: 0.24  (n=315)
    2026-03-20: 0.29  (n=629)
    2026-03-21: 0.20  (n=1247)
    2026-03-22: 0.19  (n=1045)
    2026-03-23: 0.19  (n=1395)
    2026-03-24: 0.23  (n=1606)
    2026-03-25: 0.20  (n=1586)
    2026-03-26: 0.22  (n=1579)
    2026-03-27: 0.15  (n=1732)
    2026-03-28: 0.19  (n=434)
    2026-03-29: 0.15  (n=1014)
    2026-03-30: 0.21  (n=1182)
    2026-03-31: 0.24  (n=1142)


In [19]:
print("\nStarting cross-validation...")
results, val_curves, ae_curves, latent_store, anomaly_rows = run_cv(
    X_seq, y, trip_ids, seq_lens, groups)


Starting cross-validation...
15:17:27 CV start — 20 service days, 5 folds, N=22605
15:17:27 FOLD 1/5 start | train delayed 3784 | test delayed 946 | test days ['2026-03-20', '2026-03-21', '2026-03-27', '2026-03-30']


15:17:28   [Phase 1] training autoencoder...


      AE epoch  10/20 | train MSE: 0.35405 | val MSE: 0.30808


      AE epoch  20/20 | train MSE: 0.33280 | val MSE: 0.29379
15:25:49     final AE MSE train=0.33280 val=0.29379


15:25:53   training AE+MLP...


...

16:55:05   training AE+RF...


16:55:09   training AE+LR...


16:55:12   training AE+LSTM (latent sequence)...


17:03:53   training AE+GRU (latent sequence)...


17:09:02     AE+MLP      F1 0.538 ROC-AUC 0.799
17:09:02     AE+XGBoost  F1 0.596 ROC-AUC 0.823
17:09:02     AE+RF       F1 0.587 ROC-AUC 0.833
17:09:02     AE+LR       F1 0.544 ROC-AUC 0.795
17:09:02     AE+LSTM     F1 0.556 ROC-AUC 0.802
17:09:02     AE+GRU      F1 0.540 ROC-AUC 0.797
17:09:02 FOLD 5/5 done -> ./artifacts_ae_cnn/cv_fold5.pkl
17:09:02 CV complete


In [20]:
# Export AE/val curves, latent vectors, and out-of-fold anomaly scores
ae_curve_rows = []
for split in ("train", "val"):
    for fold_i, fold_vals in enumerate(ae_curves[split]):
        for epoch_i, value in enumerate(fold_vals):
            ae_curve_rows.append({"fold": fold_i + 1, "split": split,
                                   "epoch": epoch_i + 1, "mse": value})
pd.DataFrame(ae_curve_rows).to_csv(f"{artifact_dir}/ae_reconstruction.csv", index=False)
print(f"  Saved -> {artifact_dir}/ae_reconstruction.csv")

val_rows = []
for model in MODEL_NAMES:
    for split in ("train", "val"):
        for fold_i, fold_vals in enumerate(val_curves[model][split]):
            for step_i, value in enumerate(fold_vals):
                val_rows.append({"model": model, "fold": fold_i + 1,
                                 "split": split, "step": step_i + 1,
                                 "loss": value})
pd.DataFrame(val_rows).to_csv(f"{artifact_dir}/validation_curves.csv", index=False)
print(f"  Saved -> {artifact_dir}/validation_curves.csv")

np.savez_compressed(
    f"{artifact_dir}/latent_vectors.npz",
    **{f"fold{f}_{k}": v for f, store in latent_store.items()
       for k, v in store.items()})
print(f"  Saved -> {artifact_dir}/latent_vectors.npz")

anomaly_df = pd.DataFrame(anomaly_rows)
anomaly_df.to_csv(f"{artifact_dir}/anomaly_scores.csv", index=False)
print(f"  Saved -> {artifact_dir}/anomaly_scores.csv "
      f"({len(anomaly_df)} out-of-fold trip scores)")

  Saved -> ./artifacts_ae_cnn/ae_reconstruction.csv
  Saved -> ./artifacts_ae_cnn/validation_curves.csv


  Saved -> ./artifacts_ae_cnn/latent_vectors.npz
  Saved -> ./artifacts_ae_cnn/anomaly_scores.csv (22605 out-of-fold trip scores)


In [21]:
summary_df = summarize_results(results)
print_summary_table(summary_df)
out_csv = f"{artifact_dir}/results.csv"
summary_df.to_csv(out_csv, index=False)
print(f"\n  Results saved -> {out_csv}")


  RESULTS SUMMARY (mean +/- std across folds)

  AE+MLP
    accuracy    : 0.7464 +/- 0.0276
    precision   : 0.4368 +/- 0.0284
    recall      : 0.7376 +/- 0.0643
    f1          : 0.5478 +/- 0.0330
    roc_auc     : 0.8243 +/- 0.0186
    pr_auc      : 0.5927 +/- 0.0483

  AE+XGBoost
    accuracy    : 0.8097 +/- 0.0138
    precision   : 0.5391 +/- 0.0502
    recall      : 0.6369 +/- 0.0483
    f1          : 0.5814 +/- 0.0348
    roc_auc     : 0.8383 +/- 0.0144
    pr_auc      : 0.6288 +/- 0.0457

  AE+RF
    accuracy    : 0.8119 +/- 0.0126
    precision   : 0.5462 +/- 0.0511
    recall      : 0.6119 +/- 0.0551
    f1          : 0.5740 +/- 0.0353
    roc_auc     : 0.8392 +/- 0.0137
    pr_auc      : 0.6206 +/- 0.0477

  AE+LR
    accuracy    : 0.7557 +/- 0.0182
    precision   : 0.4456 +/- 0.0380
    recall      : 0.6942 +/- 0.0511
    f1          : 0.5414 +/- 0.0360
    roc_auc     : 0.8109 +/- 0.0148
    pr_auc      : 0.5643 +/- 0.0480

  AE+LSTM
    accuracy    : 0.7556 +/- 0.0194


In [22]:
plot_results(summary_df,          f"{artifact_dir}/plot_metrics.png")
plot_ae_reconstruction(ae_curves, f"{artifact_dir}/ae_reconstruction.png")
plot_validation_curves(val_curves, f"{artifact_dir}/validation_curves.png")
plot_latent_tsne(latent_store,    f"{artifact_dir}/latent_tsne.png")

  Comparison plot saved -> ./artifacts_ae_cnn/plot_metrics.png


  AE reconstruction curves saved -> ./artifacts_ae_cnn/ae_reconstruction.png


  Validation curves saved -> ./artifacts_ae_cnn/validation_curves.png
  Fold 1: t-SNE on 4000 vectors...


  Fold 2: t-SNE on 4000 vectors...


  Fold 3: t-SNE on 4000 vectors...


  Fold 4: t-SNE on 4000 vectors...


  Fold 5: t-SNE on 4000 vectors...


  t-SNE grid saved -> ./artifacts_ae_cnn/latent_tsne.png
